# Expense Analyst Agent — Walkthrough

Run **top to bottom**. Need a `.env` with `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_API_KEY`.

1. **Install + imports** — packages, then SDK / PDF / Gradio imports.
2. **Azure setup** — load keys, disable tracing, wire the chat model.
3. **Tools** — PDF text, CSV parse/validate, spending summary by category.
4. **Agent** — `ExpenseAnalystAgent` uses those tools.
5. **Tests** — one PDF run, one CSV run. Check the printed summaries.
6. **Chat** — last code cell opens Gradio. Sample files: `data/MonthlySpendingData_Placeholder.pdf`, `data/sample_transactions.csv`.

## 1. Install packages

Run this once. Installs the Agents SDK, Azure OpenAI client, PDF reader, Gradio, and pandas.

In [98]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio pandas

Resolved 75 packages in 252ms                                        
Checked 75 packages in 1ms


## Imports

Brings in dotenv, Azure OpenAI, Agents SDK, PDF reader, Gradio, and pandas. No output expected.

In [ ]:
from dotenv import load_dotenv
import os

from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
    handoff,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from pypdf import PdfReader
import gradio as gr
import pandas as pd

## 2. Load Azure keys

Reads `.env`. Both prints should be `True`. If not, stop and fix the env file.

In [100]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Wire the Azure model

Turns tracing off, then builds `azure_model` for every agent call below.

In [101]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. PDF tool

Pulls text from `data/MonthlySpendingData_Placeholder.pdf`. `extract_pdf_text` is the tool the agent can call. You should see a character-count print.

In [102]:
def read_pdf(file_path: str) -> str:
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


# AI Agent Tool
@function_tool
def extract_pdf_text(file_path: str) -> str:
    """Extract text content from a PDF file."""
    return read_pdf(file_path)


# PDF Collection
MONTHLY_EXPENSES_COLLECTION = {
    "monthly_expenses_pdf_1": {
        "filename": "MonthlySpendingData_Placeholder.pdf",
        "content": read_pdf(
            "data/MonthlySpendingData_Placeholder.pdf"
        )
    }
}


spending_data_text = MONTHLY_EXPENSES_COLLECTION[
    "monthly_expenses_pdf_1"
]["content"]

print("Loaded PDF text length:", len(spending_data_text), "characters")

Loaded PDF text length: 275 characters


## 5. CSV validation

Banks export CSVs differently — this file is a placeholder. Required columns: `date`, `description`, `amount`. Bad rows raise an error.

In [103]:
# Validate csv file
def load_validated_csv(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)

    required = {"date", "description", "amount"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    if df[["date", "amount", "description"]].isna().any().any():
        raise ValueError("CSV contains invalid or missing values")

    if df["description"].astype(str).str.strip().eq("").any():
        raise ValueError("CSV contains empty descriptions")

    return df

## 6. CSV parse tool

Wraps the validator and returns transactions as JSON. On a bad file it returns `success: False` plus the error.

In [104]:
@function_tool
def parse_csv_transactions_tool(file_path: str) -> dict:
    """Parse and validate transactions from a CSV file."""
    try:
        df = load_validated_csv(file_path)

        return {
            "success": True,
            "transaction_count": len(df),
            "transactions": df.assign(
                date=df["date"].dt.strftime("%Y-%m-%d")
            ).to_dict(orient="records")
        }

    except (
        OSError,
        ValueError,
        pd.errors.ParserError,
        UnicodeError
    ) as e:
        return {
            "success": False,
            "error": str(e)
        }

## 7. Spending summary tool

Totals spend and buckets merchants into Groceries / Dining / Transportation / Other.

In [105]:
@function_tool
def calculate_spending_summary_tool(file_path: str) -> dict:
    """Calculate total spending and expenses by category from a CSV file."""
    try:
        df = load_validated_csv(file_path)

        # Categorize transactions based on merchant descriptions
        def categorize(description: str) -> str:
            description = description.lower()

            if any(word in description for word in ["walmart", "kroger", "aldi"]):
                return "Groceries"
            elif any(word in description for word in ["starbucks", "mcdonald", "chipotle"]):
                return "Dining"
            elif any(word in description for word in ["shell", "exxon", "uber"]):
                return "Transportation"
            else:
                return "Other"

        df["category"] = df["description"].apply(categorize)

        spending_by_category = (
            df.groupby("category")["amount"]
            .sum()
            .round(2)
            .to_dict()
        )

        return {
            "success": True,
            "total_spending": round(float(df["amount"].sum()), 2),
            "transaction_count": len(df),
            "spending_by_category": spending_by_category
        }

    except (
        OSError,
        ValueError,
        pd.errors.ParserError,
        UnicodeError
    ) as e:
        return {
            "success": False,
            "error": str(e)
        }

## 8. Save tool (stub)

Placeholder only. Not attached to the agent yet — skip mentally, do not expect it to persist data.

In [108]:
@function_tool
def save_transactions_tool():
    return

## 9. Build ExpenseAnalystAgent

System prompt + three tools: PDF extract, CSV parse, spending summary. Print should say `Agent built: ExpenseAnalystAgent`.

In [ ]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are Expense Analyst Agent, You are a Financial Advizing Agent that Analyzes financial data provided by users through 
CSV files, chat inputs, PDF files. You categorizes transactions, identifies spending patterns, and summarizes expenses for the user


You are to, when prompted by the user, to list the monthly expenses provided in the pdf from the MONTHLY_EXPENSES_COLLECTION and tell them how they could better optimize their spending.

. 

"""

ExpenseAnalystAgent = Agent(
    name="ExpenseAnalystAgent",
    handoff_description="Analyzes budget and summarizes spending.",
    instructions=instructions,
    tools=[extract_pdf_text, parse_csv_transactions_tool, calculate_spending_summary_tool],
    model=azure_model,
)

print("Agent built:", ExpenseAnalystAgent.name)

#when executed with tools that can read the MONTHLY_EXPENSES_COLLECTION, the chatbot should be able to properly assess the file
#for now, its a pdf as a placeholder but we can change that later. 
# We will add more to the description of the agent we are creating the tools - Chris


Agent built: ExpenseAnalystAgent


## 10. Test — PDF

Asks the agent to read the placeholder PDF and summarize expenses. Expect a list of bills plus optimization tips.

In [109]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "Read data/MonthlySpendingData_Placeholder.pdf and summarize my expenses."
)

print(result.final_output)

Here’s a summary of your monthly expenses from the PDF:

### Monthly Expenses
- Rent: $1,000
- Gas: $200
- Groceries: $400
- Spotify: $10
- Netflix: $15
- Take-out: $50
- Car Payment: $160
- Car Insurance: $100
- Medical Insurance: $300
- Amazon Orders: $100
- Miscellaneous: $400

### Total Monthly Spending
- **$3,500**

### Money Taken Home
- **$765**

## Quick Analysis
Your largest expenses are:
1. **Rent** — $1,000
2. **Groceries** — $400
3. **Miscellaneous** — $400
4. **Medical Insurance** — $300
5. **Gas** — $200

## Ways to Optimize Spending
- **Review Miscellaneous spending ($400):** This is a high flexible category and likely the best place to cut back.
- **Trim Amazon orders ($100):** Consider setting a monthly cap or delaying non-essential purchases.
- **Reduce grocery costs ($400):** Meal planning, buying store brands, and shopping with a list could help.
- **Cut recurring subscriptions:** Spotify and Netflix are small individually, but reviewing subscriptions is always wort

## 11. Test — CSV

Asks the agent to read `data/sample_transactions.csv`. Expect a transaction count, total, and category breakdown.

In [ ]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "Read data/sample_transactions.csv")

print(result.final_output)

I read `data/sample_transactions.csv`.

Summary:
- Transactions: 25
- Total spending: $1,642.54

Spending by category:
- Other: $1,267.43
- Groceries: $210.86
- Transportation: $112.75
- Dining: $51.50

A few transactions from the file:
- 2026-10-01 — Walmart Groceries — $85.50
- 2026-10-02 — Starbucks Coffee — $6.75
- 2026-10-05 — Shell Gas Station — $38.60
- 2026-10-07 — Netflix Subscription — $15.49
- 2026-10-14 — Electric Bill — $82.35
- 2026-10-15 — Apartment Rent — $750.00
- 2026-10-20 — Internet Bill — $49.99

Quick insights:
- Your largest spending area is “Other,” largely driven by rent, bills, subscriptions, and shopping.
- Groceries are moderate at $210.86.
- Transportation and dining are relatively controlled.

Ways to optimize spending:
- Review “Other” expenses first, especially discretionary shopping like Amazon, Target, movies, and subscriptions.
- Reduce repeat small purchases like coffee and takeout; they add up over the month.
- Check whether subscriptions like Netfl

## 12. Gradio chat

Opens a chat UI in the browser. Session memory goes to `user_expenses.db`. Try: *Read data/sample_transactions.csv* or *summarize my PDF expenses*.

In [112]:
chat_session = SQLiteSession("gradio_user", "user_expenses.db")

async def chat(message, history):
    result = await Runner.run(ExpenseAnalystAgent, message, session=chat_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Notes (from Chris)

Skeleton agent: one PDF for now, tools first, memory later. Last cell is teammate comments only.

In [ ]:
# Hey guys, so this is basically the skeleton of the Expenses Agent, I formatted it in a way that it will read a single pdf file when given the tools to do so
#We will end up adding memory as well, preferably after the tools are implemented for this agent
#It may be a little thrown together in terms of the structure, but I made what I could on my own. If you guys have issues with it, we will modify it together. 
#-Chris